# Weather Data Analysis


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("WeatherAnalysis")
    .getOrCreate()
)

base_path = "data/"

In [ ]:
cities = spark.read.csv(base_path + 'cities.csv', header=True, inferSchema=True)
countries = spark.read.csv(base_path + 'countries.csv', header=True, inferSchema=True)
daily_weather = spark.read.parquet(base_path + 'daily_weather.parquet')

Exploring Data

In [ ]:
cities.show(5)
countries.show(5)
daily_weather.show(5)

+----------+----------+-----------+----------+----+----+-------------+-------------+
|station_id| city_name|    country|     state|iso2|iso3|     latitude|    longitude|
+----------+----------+-----------+----------+----+----+-------------+-------------+
|     41515|  Asadabad|Afghanistan|     Kunar|  AF| AFG|34.8660000397|71.1500045859|
|     38954|  Fayzabad|Afghanistan|Badakhshan|  AF| AFG|37.1297607616|70.5792471913|
|     41560| Jalalabad|Afghanistan| Nangarhar|  AF| AFG|34.4415269155|70.4361034738|
|     38947|    Kunduz|Afghanistan|    Kunduz|  AF| AFG|36.7279506623|68.8725296619|
|     38987|Qala i Naw|Afghanistan|   Badghis|  AF| AFG| 34.983000131|63.1332996367|
+----------+----------+-----------+----------+----+----+-------------+-------------+
only showing top 5 rows
+--------------+--------------+----+----+-----------+---------+---------+-----------+-----------+--------------------+---------+
|       country|   native_name|iso2|iso3| population|     area|  capital|capital_l

In [ ]:
# Check if rows have the same city_name and station_id
duplicate_city_station = cities.groupBy("city_name", "station_id") \
    .count() \
    .filter("count > 1") \
    .orderBy("station_id")

duplicate_city_station.show(truncate=False)

+---------+----------+-----+
|city_name|station_id|count|
+---------+----------+-----+
|Kingston |78397     |2    |
+---------+----------+-----+



In [ ]:
# Filter all rows with city_name = 'Kingston'
kingston_rows = cities.filter(cities.city_name == "Kingston")

kingston_rows.show(truncate=False)

+----------+---------+--------------+--------+----+----+-------------+--------------+
|station_id|city_name|country       |state   |iso2|iso3|latitude     |longitude     |
+----------+---------+--------------+--------+----+----+-------------+--------------+
|78397     |Kingston |Jamaica       |Kingston|JM  |JAM |17.9770766238|-76.7674337137|
|78397     |Kingston |Norfolk Island|Unknown |NF  |NFK |17.971215    |-76.792813    |
+----------+---------+--------------+--------+----+----+-------------+--------------+



In [ ]:
# Filter countries where capital = 'Kingston'
kingston_capital = countries.filter(countries.capital == "Kingston")

kingston_capital.show(truncate=False)

+--------------+--------------+----+----+----------+-------+--------+-----------+-----------+-------------------------+-------------+
|country       |native_name   |iso2|iso3|population|area   |capital |capital_lat|capital_lng|region                   |continent    |
+--------------+--------------+----+----+----------+-------+--------+-----------+-----------+-------------------------+-------------+
|Jamaica       |Jamaica       |JM  |JAM |2717991.0 |10991.0|Kingston|17.971215  |-76.792813 |Caribbean                |North America|
|Norfolk Island|Norfolk Island|NF  |NFK |2302.0    |36.0   |Kingston|17.971215  |-76.792813 |Australia and New Zealand|Oceania      |
+--------------+--------------+----+----+----------+-------+--------+-----------+-----------+-------------------------+-------------+



In [ ]:
# Check if rows have same iso2 and city_name
from pyspark.sql.functions import count

duplicates = cities.groupBy("iso2", "city_name") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1")

duplicates.show(truncate=False)

+----+---------+---+
|iso2|city_name|cnt|
+----+---------+---+
+----+---------+---+



## 1. Data Preparation

In [ ]:
# Select capital cities only from cities
from pyspark.sql.functions import col

capitals = countries.select("iso2", "capital", "region", "continent") \
    .withColumnRenamed("iso2", "country_iso2")
capital_cities_clean = cities.join(
    capitals,
    (cities.city_name == capitals.capital) &
    (cities.iso2 == capitals.country_iso2),
    "inner"
).select(
    "station_id",
    col("city_name").alias("name"),
    "iso2",
    "latitude",
    "longitude",
    "region",
    "continent"
)

In [ ]:
# Join city & country info with daily_weather
weather_capitals = daily_weather.join(
    capital_cities_clean,
    (daily_weather.station_id == capital_cities_clean.station_id) &
    (daily_weather.city_name == capital_cities_clean.name),
    "inner"
)

In [ ]:
# Select the data from the top 10 cities only
top10_cities = weather_capitals.groupBy("city_name", "iso2") \
    .agg(count("*").alias("record_count")) \
    .orderBy("record_count", ascending=False) \
    .limit(10)

top10_cities.show()

+---------+----+------------+
|city_name|iso2|record_count|
+---------+----+------------+
| Brussels|  BE|       69347|
|   Vienna|  AT|       61477|
|Stockholm|  SE|       59774|
|   Zagreb|  HR|       59084|
|   Dublin|  IE|       57154|
|     Kiev|  UA|       51816|
| Tashkent|  UZ|       51379|
|  Vilnius|  LT|       50333|
|    Vaduz|  LI|       49940|
|  Tbilisi|  GE|       49697|
+---------+----+------------+



In [ ]:
top10_data = weather_capitals.join(
    top10_cities,
    on=["city_name", "iso2"],
    how="inner"
)

In [ ]:
# Drop all the records with missing values in min_temp_c column
top10_data2 = top10_data.dropna(subset=["min_temp_c"])

In [ ]:
# Engineer new columns
from pyspark.sql.functions import year, month, dayofmonth

data = top10_data2 \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date"))

In [ ]:
# Engineer new columns
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff, when

window = Window.partitionBy("city_name", "iso2").orderBy("date")

data = data \
    .withColumn("prev_date", lag("date", 1).over(window)) \
    .withColumn("prev_temp", lag("min_temp_c", 1).over(window))

data = data.withColumn(
    "previous_day_temp",
    when(
        datediff(col("date"), col("prev_date")) == 1,
        col("prev_temp")
    ).otherwise(None)
)

+---------+----+----------+-------------------+------+----------+----------+----------+----------------+-------------+----------------+------------------+------------------+----------------------+------------------+-----------------+----------+------+------------+--------------+-------------+---------+------------+----+-----+---+-------------------+---------+-----------------+
|city_name|iso2|station_id|               date|season|avg_temp_c|min_temp_c|max_temp_c|precipitation_mm|snow_depth_mm|avg_wind_dir_deg|avg_wind_speed_kmh|peak_wind_gust_kmh|avg_sea_level_pres_hpa|sunshine_total_min|__index_level_0__|station_id|  name|    latitude|     longitude|       region|continent|record_count|year|month|day|          prev_date|prev_temp|previous_day_temp|
+---------+----+----------+-------------------+------+----------+----------+----------+----------------+-------------+----------------+------------------+------------------+----------------------+------------------+-----------------+-------

In [ ]:
# Select relevant columns
data = data.select(
    "min_temp_c",
    "city_name",
    "date",
    "season",
    "latitude",
    "longitude",
    "region",
    "continent",
    "year",
    "month",
    "day",
    "previous_day_temp"
)

In [ ]:
# Drop all the rows with NA
final_data = data.dropna()

In [ ]:
# Save final_data as csv file
final_data.coalesce(1).write.mode("overwrite").csv(
    "/content/gdrive/MyDrive/CS4225/final_data_csv",
    header=True
)

In [ ]:
# Load final_data
final_data = spark.read.csv(
    "/content/gdrive/MyDrive/CS4225/final_data_csv",
    header=True,
    inferSchema=True
)

In [ ]:
# Split train and test data
train_data = final_data.filter(col("date") < "2010-01-01")
test_data  = final_data.filter(col("date") >= "2010-01-01")

## 2. Build ML Pipeline

In [ ]:
# Encoder
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categorical_cols = ["city_name", "season", "region", "continent"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(inputCol=c + "_index", outputCol=c + "_vec")
    for c in categorical_cols
]

In [ ]:
# Assembler
from pyspark.ml.feature import VectorAssembler

numerical_cols = [
    "latitude", "longitude", "year", "month", "day", "previous_day_temp"
]

feature_cols = [c + "_vec" for c in categorical_cols] + numerical_cols

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [ ]:
# Scaler
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

In [ ]:
# Model
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="scaled_features",
    labelCol="min_temp_c"
)

In [ ]:
# Build Pipeline
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + encoders + [assembler, scaler, lr]
)

In [ ]:
# Creates the model
model = pipeline.fit(train_data)

## 3. Evaluate ML Pipeline

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="min_temp_c",
    predictionCol="prediction",
    metricName="mae"
)

In [ ]:
# MAE for training set
train_pred = model.transform(train_data)
train_mae = evaluator.evaluate(train_pred)

print("Training MAE:", train_mae)

Training MAE: 1.9934892290580155


In [ ]:
# MAE for testing set
test_pred = model.transform(test_data)
test_mae = evaluator.evaluate(test_pred)

print("Testing MAE:", test_mae)

Testing MAE: 1.9714118204022775
